# Explorative Analyse des CFRP‑Datensatzes

Initiale Untersuchung des CFRP‑Datensatzes aus dem SmartManuAD‑Repository

**Use case:** unüberwachte Anomalieerkennung auf Sensordaten aus der Fertigung von carbonfaserverstärkten Kunststoffen

Ziel des Notebooks:
1. Rohdaten strukturiert einzulesen (`.npz` file)
2. Untersuchen von: shape, dtypes, class balance, missing values, basic statistics.
3. Merkmalsverteilungen sowie Korrelationen zwischen den Sensoren visualisieren
4. Datensplit 70/20/10 train/val/test erzeugen.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_cfrp, split_70_20_10
from src.mlflow_utils import load_config

config = load_config(PROJECT_ROOT / "config.yaml")
config

## 1. Daten herunterladen

The SmartManuAD repository links to external sources rather than hosting raw data. Follow the repo's instructions to obtain `CFRP.npz` and place it in `data/raw/`.

In [ ]:
raw_path = PROJECT_ROOT / config["dataset"]["raw_path"]
X, y = load_cfrp(raw_path)
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)

## 2. Grundlegende properties

In [ ]:
df = pd.DataFrame(X, columns=[f"f{i:02d}" for i in range(X.shape[1])])
df["label"] = y

print("Class balance:")
print(df["label"].value_counts(normalize=True).rename("share"))

print("\nMissing values per column (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nDescribe (features only):")
df.drop(columns="label").describe().T.head(10)

## 3. Visuelle Analyse

In [ ]:
feature_cols = [c for c in df.columns if c != "label"]
n_show = min(12, len(feature_cols))
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, col in zip(axes.ravel(), feature_cols[:n_show]):
    sns.histplot(data=df, x=col, hue="label", bins=50, stat="density",
                 common_norm=False, ax=ax, element="step")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[feature_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, cbar_kws={"shrink": 0.7})
plt.title("Feature correlation matrix")
plt.show()

## 4. 70 / 20 / 10 split

In [ ]:
seed = config["project"]["random_seed"]
stratify = config["split"]["stratify_on_label"]

X_train, y_train, X_val, y_val, X_test, y_test = split_70_20_10(
    X, y, seed=seed, stratify=stratify
)

split_summary = pd.DataFrame(
    {
        "n": [len(y_train), len(y_val), len(y_test)],
        "share": [len(y_train) / len(y), len(y_val) / len(y), len(y_test) / len(y)],
        "pos_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "val", "test"],
)
split_summary